In [1]:
import dspy
from pydantic import BaseModel
from ftplib import FTP

In [2]:
test_accessions = [
    "GSE174188",
    "GSE209912",
    "GSE188367",
    "GSE136103"
]

In [3]:
def get_geo_ftp_path(accession: str) -> str:
    """
    Return the FTP directory for a GEO accession (GSE or GSM).
    """
    prefix = accession[:3]     # GSE or GSM
    number = accession[3:]
    chunk = prefix + number[:-3] + "nnn"

    # the ftp site stores series and samples in directories named by the accession number with the last three digits replaced by 'nnn'
    if prefix == "GSE":
        return f"/geo/series/{chunk}/{accession}/suppl/"
    elif prefix == "GSM":
        return f"/geo/samples/{chunk}/{accession}/suppl/"
    else:
        raise ValueError("Only GSE or GSM supported")


In [6]:
import re
import warnings

def list_geo_files(accession: str):

    ftp = FTP("ftp.ncbi.nlm.nih.gov")
    ftp.login()

    path = get_geo_ftp_path(accession)
    try:
        ftp.cwd(path)
    except:
        try:
            path = re.sub(r"suppl/$", "", path)
            ftp.cwd(path)
            warnings.warn(f"No supplementary files for: {accession}")
        except:
            raise FileNotFoundError(f"Could not find FTP path: {path}")

    files = ftp.nlst()
    ftp.quit()
    return files

In [10]:
import tempfile
import os

tmpdir = tempfile.mkdtemp()

def download_geo_supp_file(accession, file_name, output_dir: str):
    ftp = FTP("ftp.ncbi.nlm.nih.gov")
    ftp.login()
    
    path = get_geo_ftp_path(accession)
    try:
        ftp.cwd(path)
    except:
        try:
            path = re.sub(r"suppl/$", "", path)
            ftp.cwd(path)
            warnings.warn(f"No supplementary files for: {accession}")
        except:
            raise FileNotFoundError(f"Could not find FTP path: {path}")

    local_file_path = os.path.join(output_dir, file_name)
    with open(local_file_path, "wb") as f:
        try:
            ftp.retrbinary(f"RETR {file_name}", f.write)
        except Exception as e:
            ftp.quit()
            raise e

    ftp.quit()
    return local_file_path

In [ ]:
import tarfile

def list_tar_contents(file_name):
     with tarfile.open(file_name, "r:*") as tar:
        for member in tar.getmembers():
            print(member.name)

def unpack_tar_file(tar_file_path, output_dir):
    with tarfile.open(tar_file_path, "r") as tar:
        tar.extractall(path=output_dir)

In [14]:
file_lists = {acc: list_geo_files(acc) for acc in test_accessions}
file_lists

C:\Users\David\AppData\Local\Temp\ipykernel_11560\2355454777.py:16: UserWarning: No supplementary files for: GSE174188
  warnings.warn(f"No supplementary files for: {accession}")


{'GSE174188': ['matrix', 'miniml', 'soft'],
 'GSE209912': ['GSE209912_counts.mtx.gz',
  'GSE209912_readme.xls',
  'GSE209912_barcodes.csv.gz',
  'GSE209912_metadata.csv.gz',
  'GSE209912_symbols.csv.gz'],
 'GSE188367': ['GSE188367_atac_tf_counts.tar.gz',
  'filelist.txt',
  'GSE188367_RAW.tar'],
 'GSE136103': ['filelist.txt', 'GSE136103_RAW.tar']}

In [ ]:
tar_file = "GSE188367_RAW.tar"

accession = "GSE188367"

download_geo_supp_file(accession, tar_file, tmpdir)
list_tar_contents(tmpdir + "/" + tar_file)

GSM5678317_BM-Old4_counts.tar.gz
GSM5678318_BM-Old5_counts.tar.gz
GSM5678319_BM-UPN01_counts.tar.gz
GSM5678320_BM-UPN02_counts.tar.gz
GSM5678321_BM-UPN03_counts.tar.gz
GSM5678322_BM-UPN04_counts.tar.gz
GSM5678323_BM-UPN06_counts.tar.gz
GSM5678324_BM-UPN11_counts.tar.gz
GSM5678325_BM-UPN12_counts.tar.gz
GSM5678326_CD34-Old4_counts.tar.gz
GSM5678327_CD34-Old5_counts.tar.gz
GSM5678328_CD34-UPN01_counts.tar.gz
GSM5678329_CD34-UPN02_counts.tar.gz
GSM5678330_CD34-UPN03_counts.tar.gz
GSM5678331_CD34-UPN04_counts.tar.gz
GSM5678332_CD34-UPN06_counts.tar.gz
GSM5678333_CD34-UPN11_counts.tar.gz
GSM5678334_CD34-UPN12_counts.tar.gz
GSM5678335_CTR07_counts.tar.gz
GSM5678336_CTR08_counts.tar.gz
GSM5678337_CTR10_counts.tar.gz
GSM5678338_HRI07_counts.tar.gz
GSM5678339_HRI08_counts.tar.gz
GSM5678340_HRI10_counts.tar.gz
